# 34장 · 검색 Tool을 조건에 따라 반복하기

도서관 대출 질문은 검색해야 하지만 주차 요금은 정해 둔 자료의 검색 범위 밖입니다. 모든 요청이 검색 노드를 통과하는 흐름과 도구를 선택할 때만 검색하는 흐름을 한 질문씩 비교합니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 langgraph 1.2.16입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.

## 도구를 고르지 않은 상태에서 멈추기

제공 그래프는 판단을 돌리는 자리에 세 낱말을 검사하는 로컬 규칙을 둡니다. model에서 tool로 간 경우에만 rounds가 늘고 다시 model로 돌아옵니다. 실습의 한도는 검색 한 차례입니다. 생성 모델의 실제 도구 선택 성공률을 재는 예제는 아닙니다.



**볼 곳**　‘모델 판단’에서 갈라지는 두 경우를 비교하세요. 위쪽은 ‘그대로 종료’, 아래쪽은 ‘검색 도구’입니다.

**애니메이션의 의미**　도구를 고르지 않은 경우 검색 호출은 0번입니다.


### 셀 01 · 제공 셀 · 선택 검색 경로 연결하기

decide에서 finish를 돌려주는 두 조건을 찾아보세요. 주차 요금 한 건을 넣습니다.

In [1]:
records = [{'id': 'library-1',
  'text': '도서 대출 기간은 14일입니다.',
  'source': '가상 도서관 이용 안내',
  'page': 1},
 {'id': 'library-2',
  'text': '도서관은 월요일에 휴관합니다.',
  'source': '가상 도서관 이용 안내',
  'page': 2},
 {'id': 'library-3',
  'text': '무인 반납함은 24시간 이용할 수 있습니다.',
  'source': '가상 도서관 이용 안내',
  'page': 3}]
terms = ("대출", "휴관", "반납")

def retrieve(question, k=1):
    query = [int(term in question) for term in terms]
    if not any(query):
        return []
    def distance(row):
        vector = [int(term in row["text"]) for term in terms]
        return sum((a-b)**2 for a, b in zip(query, vector)), row["id"]
    return sorted(records, key=distance)[:k]

from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class Loop(TypedDict):
    question: str
    rounds: int
    documents: list[dict]

def decide(state):
    if state["rounds"] >= 1:
        return "finish"
    if any(term in state["question"] for term in terms):
        return "search"
    return "finish"

def run_tool(state):
    return {"documents": retrieve(state["question"], 1), "rounds": state["rounds"] + 1}

builder = StateGraph(Loop)
builder.add_node("model", lambda state: {})
builder.add_node("tool", run_tool)
builder.add_edge(START, "model")
builder.add_conditional_edges("model", decide, {"search": "tool", "finish": END})
builder.add_edge("tool", "model")
agent = builder.compile()

request = {'question':'주차 요금은 얼마인가요?', 'rounds':0, 'documents':[]}
print(agent.invoke(request)['rounds'])

0


0은 검색을 실행하지 않았다는 기록입니다. 검색을 했지만 결과가 빈 경우도 문서 목록은 []이므로 목록만 봐서는 두 상황을 구별하기 어렵습니다.

### 셀 02 · 같은 주차 질문을 필수 검색 노드에 보내기

이번에는 분기 없이 검색을 거치는 실제 그래프입니다. 한 요청의 검색 횟수와 목록을 함께 읽으세요.

In [2]:
fixed = StateGraph(Loop)
fixed.add_node("search", run_tool)
fixed.add_edge(START, "search")
fixed.add_edge("search", END)
mandatory = fixed.compile()
fixed_result = mandatory.invoke(request)
print(fixed_result["rounds"], fixed_result["documents"])

1 []


1과 빈 목록이 나옵니다. 같은 질문에서 선택 검색은 위 셀처럼 0회, 필수 검색은 1회 호출 뒤 빈 결과였습니다.

## 검색 뒤 판단 지점으로 돌아오기

일단 선택한 도구의 결과가 상태에 들어오면 다시 판단합니다. 완료한 검색을 세지 않거나 종료 조건보다 검색 조건을 먼저 검사하면 같은 요청을 되풀이할 수 있습니다.



**볼 곳**　‘도구 요청’에서 시작해 ‘검색 결과’를 거친 뒤 ‘모델 재판단’에서 다시 판단하는 흐름을 따라가세요.

**애니메이션의 의미**　검색을 마친 뒤 모델이 다음 행동을 다시 판단합니다.


### 셀 03 · 대출 질문 한 건 추적하기

같은 초기 상태에 대출 질문을 넣어 결과와 검색 횟수를 확인합니다.

이 대출 질문에서 따라갈 길은 model → tool → model → END입니다. 뒤의 참고 그림에는 답변 작성과 호출 상한에 도달했을 때의 길까지 있습니다. 먼저 이 질문에서 실제로 지난 길만 짚어 보세요.

In [3]:
result = agent.invoke({**request, 'question':'대출 기간은 며칠인가요?'})
print(result['rounds'], result['documents'][0]['id'])

1 library-1


1회 검색에서 library-1을 찾았습니다. tool 뒤에 model로 돌아왔지만 rounds가 1이어서 두 번째 검색을 요청하지 않습니다.

### 과제 04 · 상한에 도달한 분기 확인하기

대출 질문이라도 이미 한 번 검색했다면 decide가 무엇을 반환하는지 확인하세요.

In [4]:
# 결과 상태 result를 판단 함수에 전달하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>print(decide(result))</code></pre>
<p>실행 결과</p><pre><code>finish</code></pre>
<p>finish가 돌아옵니다. 필수 검색이 서비스 규칙이면 누락을 막기 위해 고정 노드를 두고, 선택 검색을 쓴다면 생략과 상한 종료를 따로 기록하세요.</p>
</details>

## 돌아보기

검색이 반드시 필요한 팀 서비스인지 먼저 합의하세요. 한 번 더 판단할 때의 비용과 검색 누락을 확인할 사례를 함께 정해야 구조를 고를 수 있습니다.

### 참고 구현의 정상 왕복과 마지막 answer 노드



**볼 곳**　검색한 뒤 판단 지점으로 돌아오는 선과 마지막 답변으로 가는 선을 보세요.

**애니메이션의 의미**　검색 결과를 받은 에이전트는 다시 판단합니다. 추가 도구 호출이 없으면 답변 단계로 갑니다.
